In [ ]:
!git config --global user.email "tasnimkhondoker1999.com"
!git config --global user.name "mim-1999"

In [ ]:
from getpass import getpass
token = getpass('Enter GitHub token: ')

username = 'mim-1999'
repo = 'customer-churn-prediction'
!git clone https://{token}@github.com/{username}/{repo}.git
%cd customer-churn-prediction
!git pull origin main --no-rebase --no-edit

Enter GitHub token: ··········
Cloning into 'customer-churn-prediction'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 45 (delta 12), reused 18 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 377.16 KiB | 4.49 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/customer-churn-prediction
From https://github.com/mim-1999/customer-churn-prediction
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
import pandas as pd
url = 'https://raw.githubusercontent.com/mim-1999/customer-churn-prediction/main/data/processed/telco_clean.csv'
df = pd.read_csv(url)


In [ ]:
df.shape

(7043, 27)

In [ ]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,0,0,1,0,...,True,False,False,True,False,False,False,False,True,False
1,0,0,0,0,34,1,0,1,0,1,...,False,True,False,True,False,False,False,False,False,True
2,0,0,0,0,2,1,0,1,1,0,...,True,False,False,True,False,False,False,False,False,True
3,0,0,0,0,45,0,0,1,0,1,...,False,True,False,True,False,False,True,False,False,False
4,1,0,0,0,2,1,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   int64  
 5   PhoneService                             7043 non-null   int64  
 6   MultipleLines                            7043 non-null   int64  
 7   OnlineSecurity                           7043 non-null   int64  
 8   OnlineBackup                             7043 non-null   int64  
 9   DeviceProtection                         7043 non-null   int64  
 10  TechSupport                              7043 no

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
y_train.value_counts(normalize=True)


,proportion
Churn,
0,0.734647
1,0.265353


### Vanilla LogReg

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [ ]:
# Hard class predictions (0 or 1) — like ŷ
y_pred = model.predict(X_test_scaled)

# Probability predictions — like σ(z), the actual probability of churn
y_pred_proba = model.predict_proba(X_test_scaled)

In [ ]:
y_pred_proba_churn = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Probability_Churn': y_pred_proba_churn
})
comparison.head(10)

,Actual,Predicted,Probability_Churn
0,0,0,0.045222
1,0,1,0.683879
2,0,0,0.056995
3,0,0,0.406653
4,0,0,0.021562
5,0,1,0.603770
6,0,0,0.451132
7,0,0,0.131069
8,0,0,0.002838
9,1,0,0.393816


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_churn))

[[926 109]
 [163 211]]
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

ROC-AUC: 0.8418481490092743


### to improve the baseline Balanced LogReg

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced')
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced')

In [ ]:
y_pred2 = model.predict(X_test_scaled)

y_pred_proba2 = model.predict_proba(X_test_scaled)

In [ ]:
y_pred_proba_churn2 = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_churn2))

[[749 286]
 [ 81 293]]
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.8414296416853961


In [ ]:
!git add .
!git commit -m "Describe what you did this session"
!git push